<a href="https://colab.research.google.com/github/MengOonLee/LLM/blob/main/References/LangChain/LangChainAcademy/LangChain/Foundation/CreateAgent/ipynb/1.2_web_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%bash
apt install -y zstd pciutils lshw
curl -fsSL https://ollama.com/install.sh | sh
pip install --no-cache-dir -qU \
    langchain langgraph langchain-core \
    langchain-community langchain-ollama ollama

In [ ]:
!nohup ollama serve &

In [ ]:
!ollama pull gpt-oss:20b

## Without web search

In [2]:
import warnings
warnings.filterwarnings('ignore')
import dotenv

_ = dotenv.load_dotenv(dotenv_path=".env", override=True)

In [4]:
import langchain_ollama
from langchain import agents, messages
import time

model = langchain_ollama.ChatOllama(
    model="gpt-oss:20b",
    temperature=0.1,
    max_tokens=128
)

agent = agents.create_agent(
    model=model
)

question = messages.HumanMessage(content="""
    How up to date is your training knowledge?
""")

start_time = time.time()
response = agent.invoke(input={"messages": [question]})
end_time = time.time() - start_time
print(f"Time taken: %.2fs seconds"%(end_time))

for m in response["messages"]:
    m.pretty_print()

Time taken: 3.71s seconds
================================ Human Message =================================


    How up to date is your training knowledge?

================================== Ai Message ==================================

My training data goes up to **June 2024**. I don’t have access to events, releases, or developments that occurred after that point, and I can’t browse the web in real time. If you need the very latest information, it’s a good idea to double‑check with up‑to‑date sources.


## Add web search tool

In [ ]:
from langchain import tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

web_search.invoke("Who is the current mayor of San Francisco?")

In [ ]:
agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search]
)

question = HumanMessage(content="Who is the current mayor of San Francisco?")

response = agent.invoke(
    {"messages": [question]}
)

In [ ]:
from pprint import pprint

pprint(response['messages'])

trace: https://smith.langchain.com/public/59432173-0dd6-49e8-9964-b16be6048426/r